# 2A.0 24-urenloop

Simuleert een 24-urenrace (aantal ronden in 24 uur) voor vier teams:
- Ekonomika
- VTK
- Apolloon
- LBK

Doel: onderzoeken hoe Ekonomika kan winnen door verschillende kansverdelingen
rond hun gemiddelde ronde-lengte uit te spelen.

Kenmerken:
- Ondersteunde distributies: deterministische (alle rondjes zijn hetzelfde), normale, lognormale, exponentiele, gamma
- Je kunt per team de gemiddelde ronde-tijd (in seconden) en spreiding instellen
- Monte Carlo simulatie met veel trials
- Output: win-kans per team, gemiddeld aantal rondes/rondetijden, tijdsverdelingen en opslag naar CSV
- Plotten van resultaten (matplotlib)

Gebruik:
- Pas de `teams` dictionary aan met gewenste strategieën/parameters

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
from typing import Callable, Dict, Tuple

SECONDEN_PER_UUR = 3600
RACE_SECONDEN = 24 * SECONDEN_PER_UUR


def trek_rondetijd(verdeling: str, gemiddelde: float, sd: float, aantal: int=1, rng: np.random.Generator=None):
    """Geef een steekproef (of meerdere) van rondetijden (in seconden) uit de gevraagde verdeling.
    gemiddelde en sd zijn het gemiddelde en de standaardafwijking van de rondetijd,
    ook bij verdelingen die intern andere parameters gebruiken (die rekenen we om).
    Ondersteunde verdelingen: 'deterministisch', 'normaal', 'exponentieel'
    """
    if rng is None:
        rng = np.random.default_rng()

    if verdeling == 'deterministisch':
        return np.full(aantal, gemiddelde)

    if verdeling == 'normaal':
        # Begrens naar onder op een kleine positieve waarde om niet-positieve tijden te vermijden
        steekproef = rng.normal(loc=gemiddelde, scale=sd, size=aantal)
        steekproef = np.clip(steekproef, 0.1, None)
        return steekproef

    if verdeling == 'exponentieel':
        # Exponentieel met gemiddelde = 1/lambda
        lam = 1.0 / gemiddelde
        return rng.exponential(scale=1.0/lam, size=aantal)

    raise ValueError(f"Onbekende verdeling: {verdeling}")


def ronden_in_tijd(rondetijden: np.ndarray, totale_tijd: float) -> int:
    """Tel hoeveel volledige ronden passen in totale_tijd seconden gegeven opeenvolgende rondetijden."""
    cumulatief = np.cumsum(rondetijden)
    # aantal ronden waarvan de cumulatieve tijd <= totale_tijd
    return int(np.searchsorted(cumulatief, totale_tijd, side='right'))


def simuleer_race(team_parameters: Dict[str, Dict], rng: np.random.Generator) -> Dict[str, int]:
    """Simuleer één race: genereer per team rondetijden tot 24 uur en tel ronden.
    team_parameters: {teamnaam: {'verdeling': str, 'gemiddelde': float, 'sd': float}}
    geeft terug: {teamnaam: afgewerkte_ronden}
    """
    resultaten = {}
    for team, p in team_parameters.items():
        # We genereren de rondetijden één voor één tot we RACE_SECONDEN overschrijden.
        ronden = 0
        bestede_tijd = 0.0
        # veiligheidsgrens: genereer geen miljarden steekproeven
        while bestede_tijd < RACE_SECONDEN:
            rondetijd = trek_rondetijd(p.get('verdeling','normaal'), p['gemiddelde'], p.get('sd', 0.0), aantal=1, rng=rng)[0]
            if bestede_tijd + rondetijd > RACE_SECONDEN:
                break
            bestede_tijd += rondetijd
            ronden += 1
            # optionele vroegtijdige stop als het aantal ronden absurd groot wordt (veiligheid)
            if ronden > 1000000:
                break
        resultaten[team] = ronden
    return resultaten


def voer_simulatie_uit(team_parameters: Dict[str, Dict], aantal_simulaties: int = 20000, zaad: int = 42, uitgebreid: bool = False) -> pd.DataFrame:
    rng = np.random.default_rng(zaad)
    records = []
    for t in range(aantal_simulaties):
        race = simuleer_race(team_parameters, rng)
        # noteer de winnaar (bij gelijke stand: de winst wordt verdeeld over de teams met evenveel ronden)
        max_ronden = max(race.values())
        winnaars = [team for team, ronden in race.items() if ronden == max_ronden]
        for team, ronden in race.items():
            records.append({
                'simulatie': t,
                'team': team,
                'ronden': ronden,
                'winaandeel': 1.0/len(winnaars) if team in winnaars else 0.0
            })
        if uitgebreid and (t+1) % (aantal_simulaties//5) == 0:
            print(f"{t+1}/{aantal_simulaties} simulaties klaar")
    df = pd.DataFrame.from_records(records)
    return df


def vat_resultaten_samen(df: pd.DataFrame) -> pd.DataFrame:
    # winkans = som(winaandeel)/aantal_simulaties
    aantal_simulaties = df['simulatie'].nunique()
    winkansen = df.groupby('team')['winaandeel'].sum().rename('winkans') / 1.0
    gem_ronden = df.groupby('team')['ronden'].mean().rename('gem_ronden')
    std_ronden = df.groupby('team')['ronden'].std().rename('std_ronden')
    samenvatting = pd.concat([winkansen, gem_ronden, std_ronden], axis=1)
    samenvatting['winkans'] = samenvatting['winkans'] / aantal_simulaties
    samenvatting = samenvatting.sort_values('winkans', ascending=False)

    return samenvatting


def plot_verdelingen(df: pd.DataFrame, voorvoegsel: str = 'race') -> str:
    # Histogram van het aantal ronden per team
    plt.figure(figsize=(10,6))
    teams = df['team'].unique()
    for team in teams:
        waarden = df.loc[df['team']==team, 'ronden']
        plt.hist(waarden, bins=30, alpha=0.5, label=f"{team} (gemiddelde={waarden.mean():.1f})")
    plt.xlabel('Aantal ronden in 24 uur')
    plt.ylabel('Frequentie')
    plt.title('Verdeling afgeronde ronden per team over simulaties')
    plt.legend()
    bestandsnaam = f"{voorvoegsel}_ronden_histogram.png"
    plt.tight_layout()
    plt.savefig(bestandsnaam)
    plt.close()
    return bestandsnaam


def bewaar_resultaten(df: pd.DataFrame, samenvatting: pd.DataFrame, uitvoermap: str = 'resultaten') -> None:
    os.makedirs(uitvoermap, exist_ok=True)
    df.to_csv(os.path.join(uitvoermap, 'alle_simulaties.csv'), index=False)
    samenvatting.to_csv(os.path.join(uitvoermap, 'samenvatting.csv'))


# Voorbeeldteam-parameters (rondetijd in seconden):
# Stel: gemiddelde ronde-tijd is ~ 5 minuten = 300s voor alle teams als referentie.
# Je kunt Ekonomika meerdere strategieën laten proberen door gemiddelde/sd aan te passen.

teams = {
    'Ekonomika': {'verdeling': 'normaal', 'gemiddelde': 81.0, 'sd': 10.0},
    'VTK':       {'verdeling': 'normaal', 'gemiddelde': 78.0, 'sd': 12.0},
    'Apolloon':  {'verdeling': 'exponentieel', 'gemiddelde': 77.0, 'sd': 40.0},
    'LBK':       {'verdeling': 'exponentieel', 'gemiddelde': 85.0, 'sd': 25.0},
    'VRG':      {'verdeling': 'normaal', 'gemiddelde': 121, 'sd': 10}
}

# Experiment: laat Ekonomika een agressieve strategie proberen (lager gemiddelde, hogere sd)
strategieen = {
    'basis': teams,
    'agressief': {**teams, 'Ekonomika': {'verdeling': 'normaal', 'gemiddelde': 79.0, 'sd': 25.0}},
    'conservatief': {**teams, 'Ekonomika': {'verdeling': 'normaal', 'gemiddelde': 86.0, 'sd': 5.0}},
    'lage_variantie': {**teams, 'Ekonomika': {'verdeling': 'normaal', 'gemiddelde': 85.0, 'sd': 4.0}},
}

alle_samenvattingen = {}
for naam, parameters in strategieen.items():
    print(f"Strategie wordt uitgevoerd: {naam}")
    df = voer_simulatie_uit(parameters, aantal_simulaties=50, zaad=12345)
    samenvatting = vat_resultaten_samen(df)
    print(samenvatting)
    afbeelding = plot_verdelingen(df, voorvoegsel=f'resultaten_{naam}')
    bewaar_resultaten(df, samenvatting, uitvoermap=f'resultaten_{naam}')
    alle_samenvattingen[naam] = samenvatting


print("Klaar. Resultaten zijn opgeslagen in resultaten_<strategie>/ indien niet in commentaar gezet.")

Strategie wordt uitgevoerd: basis
           winkans  gem_ronden  std_ronden
team                                      
Apolloon       0.7     1123.56   34.269734
VTK            0.3     1107.42    5.649959
Ekonomika      0.0     1066.00    3.854496
LBK            0.0     1015.46   30.230549
VRG            0.0      713.00    2.089820
Strategie wordt uitgevoerd: agressief
           winkans  gem_ronden  std_ronden
team                                      
Apolloon      0.72     1124.58   29.607700
VTK           0.26     1106.48    5.459461
Ekonomika     0.02     1092.48   12.051319
LBK           0.00     1023.92   32.117666
VRG           0.00      713.40    2.364706
Strategie wordt uitgevoerd: conservatief
           winkans  gem_ronden  std_ronden
team                                      
Apolloon      0.73     1129.50   34.202429
VTK           0.27     1107.38    5.480541
Ekonomika     0.00     1004.06    1.878015
LBK           0.00     1013.52   29.589187
VRG           0.00      712